# Chunk-trained transformer for Mixed detection (roadmap step 7)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohaaa825/Skolfie/blob/main/notebooks/chunks.ipynb)

The Mixed detector scores texts in chunks of about 60 words. The step 5 model only ever saw whole
texts, so it scores single chunks noisily. This notebook trains `transformer-chunks`: the same
DistilBERT + features model, but every training text also contributes one random chunk of about
60 words with the same label. Then it

* scores it on the usual held-out splits (to check that whole texts are not scored worse), and
* reruns the Mixed evaluation with it and compares with the step 5 model.

**Use a GPU runtime:** Runtime → Change runtime type → T4 GPU. Then **Runtime → Run all**.

Expect roughly 35–50 minutes on a T4: the GPT-2 features of the new chunks are computed once
(and cached in `My Drive/skolfie/cache/`), then training runs over the texts plus the chunks.
Keep the tab open.


## 1. Settings

In [ ]:
DATA_DIR = "/content/drive/MyDrive/skolfie/data/processed"
REPORTS_DIR = "/content/drive/MyDrive/skolfie/reports"
CACHE_DIR = "/content/drive/MyDrive/skolfie/cache"
MODELS_DIR = "/content/drive/MyDrive/skolfie/models"

REPO = "mohaaa825/Skolfie"
BRANCH = "main"

MODEL = "transformer-chunks"
BASELINE = "transformer"  # the step 5 model, for the comparison
EPOCHS = 1
BATCH_SIZE = 16           # lower to 8 if you get a CUDA out-of-memory error
MAX_TRAIN = None          # e.g. 60000 for a quicker run; None = full train split
MAX_FALSE_MIXED = 0.05    # at most this share of pure texts may be called Mixed


## 2. Check the GPU, mount Drive, clone and install

In [ ]:
import os
import subprocess

import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise SystemExit("No GPU: switch to a T4 GPU runtime (Runtime -> Change runtime type).")

from google.colab import drive

drive.mount("/content/drive")
for d in (REPORTS_DIR, CACHE_DIR, MODELS_DIR):
    os.makedirs(d, exist_ok=True)
assert os.path.exists(f"{DATA_DIR}/corpus.parquet"), "Run build_corpus.ipynb first."
assert os.path.exists(f"{DATA_DIR}/splits.parquet"), "Run evaluate.ipynb (step 3) first."


def secret(name):
    try:
        from google.colab import userdata

        return userdata.get(name)
    except Exception:
        return None


token = secret("GITHUB_TOKEN")
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if os.path.isdir("/content/Skolfie/.git"):
    subprocess.run(["git", "-C", "/content/Skolfie", "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, "/content/Skolfie"],
                   check=True)

%cd /content/Skolfie
!git log --oneline -1
!pip install -q -e ".[features]"

## 3. Train and score on the held-out splits

Progress lines show the training speed and the time left.


In [ ]:
import sys

hf_token = secret("HF_TOKEN")
env = dict(os.environ, **({"HF_TOKEN": hf_token} if hf_token else {}))


def run(cmd):
    """Run a command and stream its output into the notebook as it happens."""
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            bufsize=1, env=env)
    for line in proc.stdout:
        sys.stdout.write(line)
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}")


extra = ["--max-train", str(MAX_TRAIN)] if MAX_TRAIN else []
model = MODEL
run(["python", "-u", "scripts/evaluate.py",
     "--corpus", f"{DATA_DIR}/corpus.parquet",
     "--splits", f"{DATA_DIR}/splits.parquet",
     "--model", model,
     "--out-dir", f"{REPORTS_DIR}/{model}",
     "--cache-dir", CACHE_DIR,
     "--epochs", str(EPOCHS),
     "--batch-size", str(BATCH_SIZE),
     "--save-model", f"{MODELS_DIR}/{model}.joblib",
     "--save-predictions", *extra])

## 4. Mixed evaluation with the chunk-trained model

The same synthetic Mixed sets as before (same seed), tuned with the same cap on false Mixed calls.


In [ ]:
run(["python", "-u", "scripts/eval_mixed.py",
     "--corpus", f"{DATA_DIR}/corpus.parquet",
     "--splits", f"{DATA_DIR}/splits.parquet",
     "--model", f"{MODELS_DIR}/{MODEL}.joblib",
     "--max-false-mixed", str(MAX_FALSE_MIXED),
     "--out-dir", f"{REPORTS_DIR}/mixed-{MODEL}"])


## 5. Compare with the step 5 model


In [ ]:
import json

import pandas as pd


def load(path):
    try:
        with open(path) as f:
            return json.load(f)
    except FileNotFoundError:
        return None


rows = []
for name in (BASELINE, MODEL):
    rep = load(f"{REPORTS_DIR}/{name}/report.json")
    mix = load(f"{REPORTS_DIR}/mixed-{name}/mixed_report.json")
    row = {"model": name}
    if rep:
        for split in ("test", "test_ood_domain", "test_short"):
            row[f"AUROC {split}"] = round(rep["results"][split]["metrics"]["auroc"], 4)
    if mix:
        m = mix["rules"]["chunk"]["metrics"]
        row.update({"Mixed found": f"{m['mixed_recall']:.1%}",
                    "human→Mixed": f"{m.get('human_to_mixed_rate', float('nan')):.1%}",
                    "AI→Mixed": f"{m.get('ai_to_mixed_rate', float('nan')):.1%}",
                    "3-way macro-F1": round(m["macro_f1"], 3),
                    "span F1": round(mix["spans"]["char_f1"], 3)})
    rows.append(row)

print(pd.DataFrame(rows).set_index("model").T.to_string())


Reports are in `My Drive/skolfie/reports/transformer-chunks/` and
`My Drive/skolfie/reports/mixed-transformer-chunks/`. Tell Claude in the project thread when this
has finished; it reads them from Drive directly.
